[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/myway0101/see-read-explain-dl/blob/main/notebooks/ch11_pretrained_lm.ipynb)

# 11장 사전학습 언어 모델 · 실습 노트북
『보고, 읽고, 설명하는 딥러닝 — Colab에서 직접 만들며 배우는 딥러닝 응용』

1. BERT: 빈칸 맞히기 (11.2절)
2. GPT: 다음 단어 맞히기 (11.3절)
3. 실습 ①: 사전학습된 BERT를 미세조정해 영화평 분류하기 (11.4절)
4. 퓨샷: 예시만 보여 주고 분류하기 (11.5절)
5. 실습 ②: 기본 모델과 지시 학습 모델에 같은 질문하기 (11.8절)

**런타임 → 런타임 유형 변경 → T4 GPU**로 바꾼 뒤 실행하세요. GPU에서 모두 실행하면 15분쯤 걸립니다.
> 이 장은 Hugging Face의 `transformers` 라이브러리를 씁니다. 이 라이브러리는 PyTorch로 동작하므로, 학습 코드도 PyTorch로 씁니다.

## 0. 준비

In [2]:
import random, time
import numpy as np
import pandas as pd
import torch
from torch.utils.data import TensorDataset, DataLoader
import transformers
from transformers import (pipeline, AutoTokenizer, AutoConfig,
                          AutoModelForSequenceClassification, AutoModelForCausalLM)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
transformers.set_seed(42)        # random · numpy · torch 시드를 한꺼번에 고정
print(torch.__version__, transformers.__version__, device, torch.cuda.get_device_name() if device == 'cuda' else '')

2.11.0+cu130 5.20.2 cuda Tesla T4


## 1. BERT: 빈칸 맞히기
BERT는 문장의 단어 일부를 `[MASK]`로 가리고, 앞뒤 문맥으로 가린 단어를 맞히며 사전학습했습니다(마스크 언어 모델).
한국어 문서로 사전학습한 `klue/bert-base`에 빈칸 문장을 넣어 봅니다.

In [3]:
fill = pipeline('fill-mask', model='klue/bert-base', device=device)
for s in ['대한민국의 수도는 [MASK]입니다.', '이 영화 정말 [MASK] 추천합니다.', '배가 고파서 밥을 [MASK].']:
    print(s, [(r['token_str'], round(r['score'], 3)) for r in fill(s, top_k=5)])

config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  445MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: klue/bert-base
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

대한민국의 수도는 [MASK]입니다. [('서울', 0.597), ('광화문', 0.063), ('부산', 0.047), ('평양', 0.044), ('인천', 0.032)]
이 영화 정말 [MASK] 추천합니다. [('강력', 0.6), ('적극', 0.139), ('정말', 0.039), ('너무', 0.03), ('완전', 0.021)]
배가 고파서 밥을 [MASK]. [('.', 0.667), ('[UNK]', 0.056), ('준다', 0.039), ('한다', 0.032), ('산다', 0.03)]


## 2. GPT: 다음 단어 맞히기
GPT 계열은 앞의 단어들만 보고 **다음 단어**를 맞히며 사전학습합니다. 작은 공개 모델 `Qwen2.5-0.5B`(파라미터 약 5억 개)로 확인합니다.
다음 단어 후보의 확률을 보고, 가장 확률이 높은 단어를 이어 붙이기를 되풀이해 문장을 만듭니다.

In [4]:
lm_name = 'Qwen/Qwen2.5-0.5B'
tok_lm = AutoTokenizer.from_pretrained(lm_name)
base = AutoModelForCausalLM.from_pretrained(lm_name, dtype=torch.float16 if device == 'cuda' else torch.float32).to(device)
print('파라미터:', sum(p.numel() for p in base.parameters()))

prompt = 'The capital of France is'
ids = tok_lm(prompt, return_tensors='pt').input_ids.to(device)
with torch.no_grad():
    prob = torch.softmax(base(ids).logits[0, -1].float(), dim=-1)    # 마지막 자리의 다음 단어 확률
top = torch.topk(prob, 5)
print(prompt, '→', [(tok_lm.decode([int(i)]), round(v.item(), 3)) for v, i in zip(top.values, top.indices)])

out = base.generate(ids, max_new_tokens=20, do_sample=False)          # 가장 확률 높은 단어를 20번 이어 붙임
print(tok_lm.decode(out[0], skip_special_tokens=True))

config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

파라미터: 494032768
The capital of France is → [(' Paris', 0.319), (' ______', 0.114), (' ____', 0.064), (' __', 0.055), (':\n', 0.05)]
The capital of France is Paris. It is the largest city in Europe and the second largest in the world. It is also


## 3. 실습 ①: 사전학습된 BERT로 영화평 분류하기
네이버 영화평 데이터(NSMC)는 영화평 20만 개에 긍정(1)·부정(0)이 붙어 있습니다. Colab 무료 GPU에서 몇 분 안에 학습하도록 학습 5,000개, 테스트 2,000개만 씁니다.

In [5]:
!wget -q https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt
!wget -q https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt

train_df = pd.read_csv('ratings_train.txt', sep='\t').dropna().sample(5000, random_state=42)
test_df = pd.read_csv('ratings_test.txt', sep='\t').dropna().sample(2000, random_state=42)
print(len(train_df), len(test_df), '긍정 비율:', train_df.label.mean().round(3))
train_df.head()

5000 2000 긍정 비율: 0.492


,id,document,label
128076,7865795,원본이 최고,1
65517,5417631,스릴감과 훈훈함이 있는 영화.,1
85048,8357466,굉장히 저평가되는 영화중 하나라고 생각함,1
118192,8252946,정말영화같은이야기 영화여서 영화같은이야기가 좋다,1
92368,7800452,계기도없는데 이상하다,0


BERT의 토크나이저는 단어를 **서브워드**(9.9절)로 잘게 나눕니다. 그래서 사전에 없는 단어도 조각으로 나뉘어 들어갑니다.

In [6]:
tok = AutoTokenizer.from_pretrained('klue/bert-base')
print('사전 크기(서브워드 조각 수):', len(tok))
for s in ['재밌어요', '재미없네요', '생각보다 별로예요', '나는 밥을 먹었다']:
    print(s, '→', tok.tokenize(s))

def make_loader(df, shuffle):
    enc = tok(list(df.document), padding='max_length', truncation=True, max_length=64, return_tensors='pt')
    ds = TensorDataset(enc.input_ids, enc.attention_mask, torch.tensor(df.label.values))
    return DataLoader(ds, batch_size=32, shuffle=shuffle)

train_loader = make_loader(train_df, shuffle=True)
test_loader = make_loader(test_df, shuffle=False)
print('한 배치:', next(iter(train_loader))[0].shape)

사전 크기(서브워드 조각 수): 32000
재밌어요 → ['재밌', '##어요']
재미없네요 → ['재미없', '##네', '##요']
생각보다 별로예요 → ['생각', '##보', '##다', '별로', '##예', '##요']
나는 밥을 먹었다 → ['나', '##는', '밥', '##을', '먹', '##었', '##다']
한 배치: torch.Size([32, 64])


학습 함수를 만듭니다. 케라스의 `fit()`이 해 주던 일(순전파 → 손실 → 역전파 → 가중치 갱신)을 PyTorch에서는 직접 씁니다.
비교를 위해 같은 구조의 BERT를 두 번 학습합니다.
- **사전학습된 BERT**: `from_pretrained`로 사전학습한 가중치에서 출발
- **처음부터 학습한 BERT**: `from_config`로 구조만 같고 가중치는 무작위

In [7]:
def evaluate(model):
    model.eval(); correct = 0
    with torch.no_grad():
        for ids, mask, y in test_loader:
            pred = model(input_ids=ids.to(device), attention_mask=mask.to(device)).logits.argmax(-1)
            correct += (pred.cpu() == y).sum().item()
    return correct / len(test_loader.dataset)

def train(model, epochs=3, lr=2e-5):
    model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    for ep in range(1, epochs + 1):
        model.train(); t = time.time(); total = 0
        for ids, mask, y in train_loader:
            out = model(input_ids=ids.to(device), attention_mask=mask.to(device), labels=y.to(device))
            out.loss.backward()            # 역전파
            opt.step(); opt.zero_grad()    # 가중치 갱신
            total += out.loss.item()
        print(f'epoch {ep}: loss {total / len(train_loader):.4f}, test acc {evaluate(model):.4f}, {time.time() - t:.0f}s')
    return model

In [8]:
transformers.set_seed(42)
bert = AutoModelForSequenceClassification.from_pretrained('klue/bert-base', num_labels=2)
print('파라미터:', sum(p.numel() for p in bert.parameters()), '/ 인코더 층 수:', bert.config.num_hidden_layers)
bert = train(bert)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


파라미터: 110618882 / 인코더 층 수: 12
epoch 1: loss 0.3913, test acc 0.8585, 58s
epoch 2: loss 0.2334, test acc 0.8650, 64s
epoch 3: loss 0.1302, test acc 0.8590, 63s


In [9]:
transformers.set_seed(42)
config = AutoConfig.from_pretrained('klue/bert-base', num_labels=2)
scratch = AutoModelForSequenceClassification.from_config(config)       # 같은 구조, 무작위 가중치
scratch = train(scratch)

epoch 1: loss 0.6979, test acc 0.5035, 63s
epoch 2: loss 0.5956, test acc 0.7225, 63s
epoch 3: loss 0.4390, test acc 0.7750, 64s


9장에서 사전에 없어 빈 목록이 되던 문장들을 미세조정한 BERT에 넣어 봅니다.

In [10]:
def predict(model, sentences):
    model.eval()
    enc = tok(sentences, padding=True, return_tensors='pt').to(device)
    with torch.no_grad():
        p = torch.softmax(model(**enc).logits, -1)[:, 1]    # 긍정일 확률
    for s, v in zip(sentences, p):
        print(f'{s:<16s} 긍정 확률 {v.item():.3f}')

predict(bert, ['재밌어요', '재미없네요', '생각보다 별로예요', '배우들 연기가 정말 좋았다', '시간이 아까운 영화'])

재밌어요             긍정 확률 0.990
재미없네요            긍정 확률 0.004
생각보다 별로예요        긍정 확률 0.002
배우들 연기가 정말 좋았다   긍정 확률 0.992
시간이 아까운 영화       긍정 확률 0.003


## 4. 퓨샷: 예시만 보여 주고 분류하기
가중치를 전혀 바꾸지 않고, 프롬프트에 예시 몇 개만 보여 주어 같은 분류를 시켜 봅니다(11.5절).
`Qwen2.5-0.5B` 기본 모델에 리뷰를 주고, 다음 단어로 `긍정`과 `부정` 중 어느 쪽의 확률이 더 높은지 봅니다.

In [11]:
head = '다음 영화 리뷰가 긍정인지 부정인지 답하세요.\n\n'
shots = [('단연 최고라고 할수있지', '긍정'), ('유치하고 지루하고 잠이 왔다', '부정'),
         ('많은 감동을 준 드라마이다.', '긍정'), ('성룡영화중 최악인듯 ㅋㅋ', '부정')]   # NSMC 학습 데이터에서 고른 예시 4개
few = ''.join(f'리뷰: {r}\n감정: {l}\n\n' for r, l in shots)

def answer_logprob(prompt, answer):
    n = tok_lm(prompt, return_tensors='pt').input_ids.shape[1]
    ids = tok_lm(prompt + answer, return_tensors='pt').input_ids.to(device)
    with torch.no_grad():
        lp = torch.log_softmax(base(ids).logits[0, :-1].float(), -1)
    return sum(lp[i - 1, ids[0, i]].item() for i in range(n, ids.shape[1]))   # 답 부분의 로그 확률 합

def classify(review, examples):
    prompt = head + examples + f'리뷰: {review}\n감정:'
    return 1 if answer_logprob(prompt, ' 긍정') > answer_logprob(prompt, ' 부정') else 0

sample = test_df.head(200)
print('200개 중 긍정 비율:', sample.label.mean())
for name, ex in [('제로샷 (예시 0개)', ''), ('퓨샷 (예시 4개)', few)]:
    acc = np.mean([classify(r, ex) == y for r, y in zip(sample.document, sample.label)])
    print(f'{name}: {acc:.3f}')

200개 중 긍정 비율: 0.51
제로샷 (예시 0개): 0.555
퓨샷 (예시 4개): 0.705


In [12]:
# 같은 200개에서 미세조정한 BERT의 정확도
enc = tok(list(sample.document), padding=True, truncation=True, max_length=64, return_tensors='pt').to(device)
bert.eval()
with torch.no_grad():
    pred = bert(**enc).logits.argmax(-1).cpu().numpy()
print(f'미세조정한 BERT: {np.mean(pred == sample.label.values):.3f}')

미세조정한 BERT: 0.805


## 5. 실습 ②: 기본 모델과 지시 학습 모델에 같은 질문하기
`Qwen2.5-0.5B`는 다음 단어 맞히기로만 학습한 **기본(base) 모델**이고, `Qwen2.5-0.5B-Instruct`는 같은 모델을 질문-답변 예시로 더 학습한 **지시 학습(instruct) 모델**입니다(11.6절).
같은 질문을 넣어 답이 어떻게 다른지 봅니다.

In [13]:
chat_name = 'Qwen/Qwen2.5-0.5B-Instruct'
tok_chat = AutoTokenizer.from_pretrained(chat_name)
chat = AutoModelForCausalLM.from_pretrained(chat_name, dtype=torch.float16 if device == 'cuda' else torch.float32).to(device)

def ask_base(q, n=60):
    ids = tok_lm(q, return_tensors='pt').input_ids.to(device)
    out = base.generate(ids, max_new_tokens=n, do_sample=False)
    return tok_lm.decode(out[0, ids.shape[1]:], skip_special_tokens=True)

def ask_chat(q, n=60):
    msgs = [{'role': 'system', 'content': '한두 문장으로 짧게 답하세요.'},   # 모델에게 주는 지시
            {'role': 'user', 'content': q}]
    enc = tok_chat.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt', return_dict=True).to(device)
    out = chat.generate(**enc, max_new_tokens=n, do_sample=False)      # 대화 형식(채팅 템플릿)으로 감싸서 질문
    return tok_chat.decode(out[0, enc['input_ids'].shape[1]:], skip_special_tokens=True)

for q in ['대한민국의 수도는 어디인가요?', '사과 3개에 2개를 더하면 몇 개인가요?', '물은 몇 도에서 끓나요?', '봄 다음에 오는 계절은 무엇인가요?']:
    print('질문:', q)
    print('[기본 모델]', ask_base(q).strip().replace('\n', ' / '))
    print('[지시 학습]', ask_chat(q).strip().replace('\n', ' / '))
    print()

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

질문: 대한민국의 수도는 어디인가요?
[기본 모델] - 2020년 11월 15일 / 대한민국의 수도는 어디인가요? / 대한민국의 수도는 어디인가요? 2020년 11월 15일 / 대한민국의
[지시 학습] 대한민국의 수도는 서울입니다.

질문: 사과 3개에 2개를 더하면 몇 개인가요?
[기본 모델] - 2021-03-20 / 사과 3개에 2개를 더하면 몇 개인가요? / 사과 3개에 2개를 더하면 몇 개인가요? / 사과 3개에 2개를 더
[지시 학습] 5 개

질문: 물은 몇 도에서 끓나요?
[기본 모델] - 2019년 10월 15일 / 2019년 10월 15일 10:00 / 2019년 10월 15일 10:00 / 2019
[지시 학습] 물은 대부분의 경우 100도에서 끓습니다.

질문: 봄 다음에 오는 계절은 무엇인가요?
[기본 모델] - 2021년 1월 1일 / 2021년 1월 1일 / 2021년 1월 1일 / 2021년 1월 1일 / 2021년 1월 1
[지시 학습] 봄 다음에 오는 계절은 여름입니다.



## 실습
1. **학습 데이터 수**: 사전학습된 BERT를 학습 데이터 500개로만 미세조정하면 정확도가 얼마나 될까요? 처음부터 학습한 BERT(5,000개)와 비교해 보세요.
2. **퓨샷 예시 수**: 예시를 2개(긍정 1, 부정 1)로 줄이면 퓨샷 정확도가 어떻게 바뀔까요?
3. **내 질문**: `ask_base`와 `ask_chat`에 직접 만든 질문을 넣어 보세요.

먼저 직접 해 본 뒤, 아래 예시 답안과 비교해 보세요.

In [14]:
# 직접 작성해 보세요


### 예시 답안

In [15]:
# 1. 학습 데이터 500개로 미세조정
transformers.set_seed(42)
train_loader = make_loader(train_df.head(500), shuffle=True)
small = AutoModelForSequenceClassification.from_pretrained('klue/bert-base', num_labels=2)
small = train(small)
train_loader = make_loader(train_df, shuffle=True)      # 원래대로 되돌린다

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


epoch 1: loss 0.5892, test acc 0.7840, 14s
epoch 2: loss 0.3481, test acc 0.7985, 15s
epoch 3: loss 0.2026, test acc 0.8145, 13s


In [16]:
# 2. 퓨샷 예시 2개
few2 = ''.join(f'리뷰: {r}\n감정: {l}\n\n' for r, l in shots[:2])
acc = np.mean([classify(r, few2) == y for r, y in zip(sample.document, sample.label)])
print(f'퓨샷 (예시 2개): {acc:.3f}')

퓨샷 (예시 2개): 0.680


In [17]:
# 3. 내 질문
q = '딥러닝을 한 문장으로 설명해 주세요.'
print('[기본 모델]', ask_base(q).strip().replace('\n', ' / '))
print('[지시 학습]', ask_chat(q).strip().replace('\n', ' / '))

[기본 모델] 문장: "이제는 빨리 빠져나가야겠군요."
[지시 학습] 딥러닝은 컴퓨터 기술에서 사용되는 분류 및 클러스터링 알고리즘입니다.
